# 🏗️ 기본 세팅

## MongoDB 서버 및 Python 패키지 설치

In [1]:
# 1. MongoDB 공식 GPG 키 및 리포지토리 등록
!curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | sudo gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg
!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list

# 2. 패키지 목록 업데이트 및 설치
!apt-get update
!apt-get install -y mongodb-org > /dev/null

# 3. MongoDB 서비스 실행
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db

# 4. 파이썬용 드라이버 설치
!pip install pymongo

deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:6 https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 InRelease [3,005 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,145 kB]
Get:10 https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0/multiverse arm64 Packages [122 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http:

# ✅ 실습문제

## 사전 세팅

In [2]:
import os
from pymongo import MongoClient
from bson import json_util

client = MongoClient('mongodb://localhost:27017/')
db = client.health_db        # 원하는 DB 이름
col = db.health_records      # 원하는 컬렉션 이름

# 데이터 구성
data = [
    {"name": "David", "age": 45, "sex": "M", "systolic": 135, "diastolic": 88, "glucose": 110, "hdl": 40, "ldl": 150},
    {"name": "Alice", "age": 33, "sex": "F", "systolic": 118, "diastolic": 76, "glucose": 92, "hdl": 55, "ldl": 110},
    {"name": "Bob", "age": 52, "sex": "M", "systolic": 145, "diastolic": 95, "glucose": 160, "hdl": 38, "ldl": 170},
    {"name": "Emma", "age": 29, "sex": "F", "systolic": 109, "diastolic": 70, "glucose": 85, "hdl": 60, "ldl": 95},
    {"name": "Chris", "age": 60, "sex": "M", "systolic": 150, "diastolic": 92, "glucose": 170, "hdl": 35, "ldl": 190},
    {"name": "Sara", "age": 48, "sex": "F", "systolic": 132, "diastolic": 85, "glucose": 110, "hdl": 50, "ldl": 140}
]

# 데이터 삽입
col.insert_many(data)

print("데이터 삽입 완료!")


데이터 삽입 완료!


## 문제 1) 고혈압 환자 찾기
- 수축기 혈압(sys) 140 이상

In [5]:
# 여기에 코드를 입력하세요.

results = col.find(
    {"systolic":{"$gt":140}}
)
for doc in results:
  print(doc)

{'_id': ObjectId('6a70594e771c39d9a7839c1e'), 'name': 'Bob', 'age': 52, 'sex': 'M', 'systolic': 145, 'diastolic': 95, 'glucose': 160, 'hdl': 38, 'ldl': 170}
{'_id': ObjectId('6a70594e771c39d9a7839c20'), 'name': 'Chris', 'age': 60, 'sex': 'M', 'systolic': 150, 'diastolic': 92, 'glucose': 170, 'hdl': 35, 'ldl': 190}


## 문제 2) 대사 질환 위험군 찾기
- 공복혈당 (glucose) 125 이상
- HDL 40 미만


In [8]:
results = col.aggregate([
    {"$match":{
        "glucose":{"$gte":125},
        "hdl":{"$lt":40}
    }}
])

for doc in results:
  print(doc)

{'_id': ObjectId('6a70594e771c39d9a7839c1e'), 'name': 'Bob', 'age': 52, 'sex': 'M', 'systolic': 145, 'diastolic': 95, 'glucose': 160, 'hdl': 38, 'ldl': 170}
{'_id': ObjectId('6a70594e771c39d9a7839c20'), 'name': 'Chris', 'age': 60, 'sex': 'M', 'systolic': 150, 'diastolic': 92, 'glucose': 170, 'hdl': 35, 'ldl': 190}


## 문제 3) 심혈관 위험도가 높은 사람 3명 찾기
- LDL 점수 = ldl 값 그대로
- 혈압 점수 = 수축기 혈압 sys
- 혈당 점수 = glucose

각 환자의 위험도(risk)는 다음의 합으로 계산합니다:
`risk = ldl + systolic + glucose`

Aggregation을 사용하여
1) 필요한 필드만 선택하고  
2) 새로운 필드(risk)를 생성하여 계산한 뒤  
3) risk 값을 기준으로 내림차순 정렬하여  

가장 위험도가 높은 3명을 출력하세요.


In [12]:

pipeline = [
    {
        "$project":{
            "_id":0,
            "name":1,
            "ldl":1,
            "systolic":1,
            "glucose":1,
            "risk":{"$sum":["$ldl","$systolic","$glucose"]}
    }},{"$sort":{"risk":-1}},{"$limit":3}
]
results = list(col.aggregate(pipeline))

display(results)

[{'name': 'Chris', 'systolic': 150, 'glucose': 170, 'ldl': 190, 'risk': 510},
 {'name': 'Bob', 'systolic': 145, 'glucose': 160, 'ldl': 170, 'risk': 475},
 {'name': 'David', 'systolic': 135, 'glucose': 110, 'ldl': 150, 'risk': 395}]